# Vision Guard - Colab Run

Run the project directly from GitHub, launch the Gradio UI, scan the video first, then search with natural language and review matched frames before exporting clips.

## 1. clone repo

In [ ]:
import os
if not os.path.exists('visionguard-ai'):
    !git clone https://github.com/priyansupattanaik/visionguard-ai.git
%cd visionguard-ai
!git pull

## 2. mount drive for cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2A. set persistent model cache

In [ ]:
import os
base = '/content/drive/MyDrive/visionguard_cache'
paths = {
    'HF_HOME': f'{base}/hf',
    'TRANSFORMERS_CACHE': f'{base}/hf/transformers',
    'HUGGINGFACE_HUB_CACHE': f'{base}/hf/hub',
    'TORCH_HOME': f'{base}/torch',
    'YOLO_CONFIG_DIR': f'{base}/ultralytics',
    'ULTRALYTICS_SETTINGS': f'{base}/ultralytics/settings.json',
}
for key, value in paths.items():
    os.environ[key] = value
for key in ['HF_HOME', 'TRANSFORMERS_CACHE', 'HUGGINGFACE_HUB_CACHE', 'TORCH_HOME', 'YOLO_CONFIG_DIR']:
    os.makedirs(os.environ[key], exist_ok=True)
print('model cache:', base)

## 3. install deps

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. optional gpu check

In [ ]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'

## 5. launch app with Colab proxy

This path does not depend on Gradio share-link creation. It starts the app in the background and opens it through the Colab proxied local port.

In [ ]:
import os
import subprocess
import time
from google.colab.output import eval_js

os.environ['GRADIO_SHARE'] = '0'

proc = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

started = False
lines = []
deadline = time.time() + 120
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    print(line, end='')
    lines.append(line)
    if 'Running on local URL:' in line:
        started = True
        break

if not started:
    raise RuntimeError('Vision Guard did not start within 120 seconds. Check the logs above.')

url = eval_js('google.colab.kernel.proxyPort(7860)')
print('\nOpen this URL in the browser tab:')
print(url)

## 5A. optional live logs

In [ ]:
while True:
    line = proc.stdout.readline()
    if not line:
        break
    print(line, end='')

## 6. use the app

1. upload a CCTV video or use sample assets
2. click `step 1: scan video`
3. watch the live indexing preview until scan completes
4. type a query like `person sitting near gate`, `fight near road`, `car accident`, `white car entering`, `yellow car`
5. click `step 2: find matches`
6. review the matched-frame gallery and timestamp table
7. export only the clips you choose

If you prefer Gradio's public share-link flow instead of the Colab proxy, set `os.environ['GRADIO_SHARE'] = '1'` before launching the app.